In [ ]:
#!/usr/bin/env python3
"""Loads one run of the clustering and report steps (supp_01, supp_02): the .h5ad with joint and intra-species clusters and layouts, and its report tables.

Every .h5ad one level below the scanpy_dir of the npz_scanpy section of configs/param_configs.yaml is listed and RUN_INDEX picks one. Joint (inter-species) clusters are named by lowercase letters of their Leiden id and intra-species clusters by their id, as in every report; the significant and selected columns of the enrichment table are the species-difference calls of h5ad_scanpy.py, computed on the target inputs; the focus tables (focus_cluster_enrichment, focus_capture, focus_gravity) are its second layer, the functional focus of the probe inputs.
"""
###################################### DEPENDENCIES_AND_CONSTANTS
import anndata as ad
from IPython.display import display
import pandas as pd
from pathlib import Path
from pyprojroot import find_root, has_file
import yaml
DEFAULT_QUERY_TSV = "configs/query_configs.tsv"
YAML_PATH = "configs/param_configs.yaml"
RUN_INDEX = 0
###################################### CONFIG
PROJECT_ROOT = find_root(has_file(".here"), start=Path.cwd()).resolve()
config = yaml.safe_load((PROJECT_ROOT / YAML_PATH).read_text()) or {}
query_table = pd.read_csv(PROJECT_ROOT / config.get("dw_uniprot_seqs", {}).get("query_table", DEFAULT_QUERY_TSV), sep="\t", dtype=str)
species_names = dict(zip(query_table["label"], query_table["exact_species"].fillna(query_table["label"])))
scanpy_dir = PROJECT_ROOT / config["npz_scanpy"]["scanpy_dir"]
h5ads = sorted(scanpy_dir.glob("*/*.h5ad"))
if not h5ads: raise FileNotFoundError(f"[PYTHON-ERROR] No .h5ad found in {scanpy_dir}/<run>/; run slurms/supp_01_scanpy.sh first.")
for i, p in enumerate(h5ads): print(f"[PYTHON-INFO] {'>' if i == RUN_INDEX else ' '} [{i}] {p.relative_to(scanpy_dir)}")

In [ ]:
###################################### LOAD
def letter(i: int) -> str:
    """Name a 0-based joint cluster id by lowercase letters, spreadsheet-style (0 -> a, 26 -> aa).

    Args:
        i (int): 0-based Leiden cluster id.

    Returns:
        str: Letter name.
    """
    return (letter(i // 26 - 1) if i >= 26 else "") + chr(97 + i % 26)
adata = ad.read_h5ad(h5ads[RUN_INDEX])
run_dir, param_tag = h5ads[RUN_INDEX].parent, adata.uns["param_tag"]
ids = adata.obs["cluster"].astype(int)
adata.obs["cluster"] = pd.Categorical(ids.map(letter), categories=[letter(i) for i in sorted(ids.unique())])
report_paths = {"enrichment": run_dir / f"species_cluster_enrichment__{param_tag}.csv", "top_clusters": run_dir / f"species_top_clusters__{param_tag}.csv", "markers": run_dir / f"channel_importance_rankings__{param_tag}.csv", "focus_enrichment": run_dir / f"focus_cluster_enrichment__{param_tag}.csv", "focus_capture": run_dir / f"focus_capture__{param_tag}.csv", "focus_gravity": run_dir / f"focus_gravity__{param_tag}.csv", "hdbscan": run_dir / "interactive" / f"hdbscan_umap3d__{param_tag}.tsv"}
report = {k: pd.read_csv(p, sep="\t" if p.suffix == ".tsv" else ",") for k, p in report_paths.items() if p.exists()}
cluster_stats = report["enrichment"].drop_duplicates("cluster").set_index("cluster") if "enrichment" in report else None
print(f"[PYTHON-INFO] {h5ads[RUN_INDEX].name}: {adata.n_obs:,} proteins, {adata.obs['cluster'].nunique()} joint clusters, layouts {', '.join(adata.obsm.keys())}")
print(f"[PYTHON-INFO] Report tables: {', '.join(f'{k} {v.shape}' for k, v in report.items())}; missing: {', '.join(k for k in report_paths if k not in report) or 'none'}")
display(pd.crosstab(adata.obs["cluster"], adata.obs["species"]).rename(columns=species_names).head())
if cluster_stats is not None: print(f"[PYTHON-INFO] Selected joint clusters: {', '.join(cluster_stats.index[cluster_stats['selected']])}")